In [104]:
from fast_borf.weighted.borf_multi import BorfPipelineBuilder
from fast_borf.classes.bag_of_receptive_fields_sax.borf_multi import BorfPipelineBuilder as BorfPipelineBuilderOld
from aeon.datasets import load_classification
from fast_borf.pipeline.to_scipy import ToScipySparse
from fast_borf.pipeline.zero_columns_remover import ZeroColumnsRemover
from fast_borf.pipeline.reshaper import ReshapeTo2D
import xarray as xr

In [105]:
X_train, y_train = load_classification("CBF", split="train")
X_test, y_test = load_classification("CBF", split="test")

In [106]:
borf_builder = BorfPipelineBuilder(
    pipeline_objects=[
        (ReshapeTo2D, {}),
        (ZeroColumnsRemover, {}),
        (ToScipySparse, {})
    ],
    contains_time_idx=False
)
borf = borf_builder.build(
    X_train
)

In [107]:
borf.fit(X_train, y_train)
out1 = borf.transform(X_train)

In [108]:
borf_builder = BorfPipelineBuilderOld(
    pipeline_objects=[
        (ReshapeTo2D, {}),
        (ZeroColumnsRemover, {}),
        (ToScipySparse, {})
    ],
)
borf = borf_builder.build(
    X_train
)

In [109]:
borf.fit(X_train, y_train)
out2 = borf.transform(X_train)

In [110]:
import numpy as np
np.allclose(out2.todense(), out1.todense())

True

In [111]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from lightgbm import LGBMClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import f1_score

In [198]:
from irregular_ts.data_utils import data_new_folder
import xarray as xr
df = xr.open_dataset(data_new_folder() / "Garment.h5", engine="my_engine")["data"]
y, split = df.irr.get_task_target_and_split()
X, _ = df.irr.to_dense(
    concatenate_time=True,
    normalize_time=True,
)
train_idxs, test_idxs = split == "train", split == "test"
X_train, y_train = X[train_idxs], y[train_idxs]
X_test, y_test = X[test_idxs], y[test_idxs]
X_train.shape

(18, 10, 59)

In [199]:
borf_builder = BorfPipelineBuilder(
    pipeline_objects=[
        (ReshapeTo2D, {}),
        (ZeroColumnsRemover, {}),
        (ToScipySparse, {})
    ],
    contains_time_idx=True,
    min_window_to_signal_std_ratio=0.15,
    n_jobs=-1
)
borf = borf_builder.build(
    X_train
)

In [200]:
pipe = make_pipeline(
    borf,
    FunctionTransformer(lambda x: np.arcsinh(x)),
    RidgeClassifierCV()
)

In [201]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
f1_score(y_test, y_pred, average="macro")

0.7777777777777777

In [202]:
X, _ = df.irr.to_dense(
    concatenate_time=False,
    normalize_time=True,
)
train_idxs, test_idxs = split == "train", split == "test"
X_train, y_train = X[train_idxs], y[train_idxs]
X_test, y_test = X[test_idxs], y[test_idxs]
X_train.shape
borf_builder = BorfPipelineBuilderOld(
    pipeline_objects=[
        (ReshapeTo2D, {}),
        (ZeroColumnsRemover, {}),
        (ToScipySparse, {})
    ],
    # contains_time_idx=True,
    min_window_to_signal_std_ratio=0.15,
    n_jobs=-1
)
borf = borf_builder.build(
    X_train
)
pipe = make_pipeline(
    borf,
    FunctionTransformer(lambda x: np.arcsinh(x)),
    RidgeClassifierCV()
)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
f1_score(y_test, y_pred, average="macro")

0.7777777777777777